<a href="https://colab.research.google.com/github/vaisanthragavc2025-pixel/vaisanth-EDA/blob/main/25BAI0145_EXP5_04.08.2026/FraudTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import os
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Step 1: Load data file
test_path = '/content/fraudTest.csv'
df_test = pd.read_csv(test_path) if os.path.exists(test_path) else pd.DataFrame({'amt': [20.0, 95.0, 1500.0], 'age': [30, 22, 55]})

# Calculate 'age' for df_test if 'dob' and 'trans_date_trans_time' exist
if 'dob' in df_test.columns and 'trans_date_trans_time' in df_test.columns:
    df_test['trans_date_trans_time'] = pd.to_datetime(df_test['trans_date_trans_time'])
    df_test['dob'] = pd.to_datetime(df_test['dob'])
    df_test['age'] = (df_test['trans_date_trans_time'].dt.year - df_test['dob'].dt.year) - \
                ((df_test['trans_date_trans_time'].dt.month < df_test['dob'].dt.month) | \
                 ((df_test['trans_date_trans_time'].dt.month == df_test['dob'].dt.month) & \
                  (df_test['trans_date_trans_time'].dt.day < df_test['dob'].dt.day)))
else:
    # Fallback if 'dob' or 'trans_date_trans_time' are missing, create a dummy 'age' column
    df_test['age'] = df_test['amt'].apply(lambda x: 30 + (x % 50)) # Example dummy age based on amt

# Step 2: Binning (Fixed ranges and Quantiles)
bins = [-1, 20, 100, 1000, float('inf')]
labels = ['Micro', 'Standard', 'High', 'Extreme']

df_test['amt_labels'] = pd.cut(df_test['amt'], bins=bins, labels=labels)
df_test['amt_quantiles'] = pd.qcut(df_test['amt'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')

print("--- fraudTest Binning Output ---")
print(df_test[['amt', 'amt_labels', 'amt_quantiles']])

# Step 3: Outlier Detection (Fixed limit and IQR)
limit_outliers_test = df_test[df_test['amt'] > 1000]
q1_ts, q3_ts = df_test['amt'].quantile(0.25), df_test['amt'].quantile(0.75)
iqr_ts = q3_ts - q1_ts
iqr_outliers_test = df_test[df_test['amt'] > (q3_ts + 3 * iqr_ts)]

print("\n--- fraudTest Outlier Output ---")
print(f"Fixed Limit Outliers Count: {len(limit_outliers_test)}")
print(f"IQR Outliers Count: {len(iqr_outliers_test)}")

# Step 4: Data Transformation (Scaling)
features_test = df_test[['amt', 'age']]

mm_test = pd.DataFrame(MinMaxScaler().fit_transform(features_test), columns=['amt_minmax', 'age_minmax'])
std_test = pd.DataFrame(StandardScaler().fit_transform(features_test), columns=['amt_zscore', 'age_zscore'])
df_test_final = pd.concat([df_test, mm_test, std_test], axis=1)

print("\n--- fraudTest Transformation Output ---")
print(df_test_final[['amt', 'amt_minmax', 'amt_zscore', 'age', 'age_minmax', 'age_zscore']].tail())

--- fraudTest Binning Output ---
           amt amt_labels amt_quantiles
0         2.86      Micro            Q1
1        29.84   Standard            Q2
2        41.28   Standard            Q2
3        60.05   Standard            Q3
4         3.19      Micro            Q1
...        ...        ...           ...
555714   43.77   Standard            Q2
555715  111.84       High            Q4
555716   86.88   Standard            Q4
555717    7.99      Micro            Q1
555718   38.13   Standard            Q2

[555719 rows x 3 columns]

--- fraudTest Outlier Output ---
Fixed Limit Outliers Count: 1583
IQR Outliers Count: 11995

--- fraudTest Transformation Output ---
           amt  amt_minmax  amt_zscore  age  age_minmax  age_zscore
555714   43.77    0.001879   -0.163467   54    0.481481    0.436520
555715  111.84    0.004868    0.270803   21    0.074074   -1.456529
555716   86.88    0.003772    0.111564   39    0.296296   -0.423957
555717    7.99    0.000307   -0.391735   55    0.49382